<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/03-neural-network-building-blocks.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Neural Network Building Blocks** {#neural-network-building-blocks}

A modern neural network can contain billions of parameters, yet most of its computation is assembled from a small vocabulary: affine transformations mix features, nonlinearities create expressive decision boundaries, embeddings turn discrete identities into learned vectors, output heads translate representations into task-specific predictions, and residual paths, gates, and normalization regulate information flow.

Learning these components separately is useful for three reasons. First, it turns an architecture diagram into an auditable sequence of tensor contracts. Second, it explains why two networks with the same input and output shapes can train very differently. Third, it makes implementation modular: a new model is often a careful rearrangement of familiar blocks rather than an entirely new algorithm.

Throughout the chapter, $B$ denotes batch size, $L$ sequence length, $D$ feature width, $V$ vocabulary size, and $K$ number of output classes. For each block, the central questions are:

1. What information does it receive and return?
2. Which axes does it mix, preserve, or normalize?
3. Which quantities are learned parameters and which are data-dependent activations?
4. What failure appears when the block is omitted, misplaced, or given incompatible shapes?

### **Artificial Neurons and Linear Transformations** {#artificial-neurons-linear-transformations}

An **artificial neuron** combines input features into one scalar and optionally passes that scalar through a nonlinear activation. For an input vector $\mathbf{x} \in \mathbb{R}^{D_{in}}$,

$$
z = \mathbf{w}^{\top}\mathbf{x} + b,
\qquad
y = \phi(z).
$$

The weights $\mathbf{w}$ determine how strongly each feature contributes, the bias $b$ shifts the response threshold, and $\phi$ determines how the pre-activation $z$ becomes an activation $y$. A useful analogy is a weighted committee: every feature casts a signed vote, the bias represents a prior preference, and the activation decides how the final score is exposed to the next layer.

A layer normally computes many neurons in parallel. With a mini-batch $X \in \mathbb{R}^{B \times D_{in}}$ and $D_{out}$ output neurons,

$$
Z = XW^{\top} + \mathbf{b},
\qquad
W \in \mathbb{R}^{D_{out} \times D_{in}},
\qquad
\mathbf{b} \in \mathbb{R}^{D_{out}}.
$$

The output has shape $[B,D_{out}]$. Every output neuron receives every input feature, which is why the operation is called **fully connected** or **dense**. Mathematically, $XW^{\top}$ is linear, while adding a nonzero bias makes the complete map affine. In everyday deep-learning language, however, `Linear` usually refers to the whole affine operation.

Geometrically, a scalar pre-activation $z=\mathbf{w}^{\top}\mathbf{x}+b$ measures signed position relative to the hyperplane $\mathbf{w}^{\top}\mathbf{x}+b=0$. A single neuron can therefore represent a linear decision boundary, but it cannot by itself express a disconnected region, an XOR relation, or a feature whose effect reverses in different contexts. Those limitations motivate composition and nonlinear activation.

<details>
<summary><strong>PyTorch: reproduce <code>nn.Linear</code> from its stored parameters</strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(7)

B, D_IN, D_OUT = 3, 4, 2
features = torch.randn(B, D_IN)
linear = nn.Linear(D_IN, D_OUT, bias=True)

# PyTorch stores weight as [D_out, D_in], so the batch uses weight.T.
manual_output = features @ linear.weight.T + linear.bias
module_output = linear(features)

assert linear.weight.shape == (D_OUT, D_IN)
assert linear.bias.shape == (D_OUT,)
assert module_output.shape == (B, D_OUT)
assert torch.allclose(manual_output, module_output)

print("input:", tuple(features.shape))
print("weight:", tuple(linear.weight.shape))
print("output:", tuple(module_output.shape))
~~~

</details>

**Application.** Linear transformations are used as classifiers, regression layers, feature projections, attention query/key/value projections, channel mixers, and the expanding and contracting maps inside Transformer feed-forward networks. Their role is not always to make a final prediction; often they rotate, combine, or resize an intermediate representation so that a later block can use it.

**Comparison summary.** A neuron returns one weighted response; a dense layer evaluates many neurons together. A bias-free map preserves the origin and is strictly linear, while a biased map is affine. Linear layers mix information across the feature axis, but without nonlinearities, stacking them does not create a genuinely nonlinear model.

### **Multilayer Perceptrons** {#multilayer-perceptrons}

A **multilayer perceptron (MLP)** alternates affine transformations and nonlinear activation functions. For one hidden layer,

$$
H = \phi(XW_1^{\top}+\mathbf{b}_1),
\qquad
O = HW_2^{\top}+\mathbf{b}_2.
$$

Here $X \in \mathbb{R}^{B \times D_{in}}$, $H \in \mathbb{R}^{B \times D_h}$, and $O \in \mathbb{R}^{B \times D_{out}}$. The hidden width $D_h$ controls how many intermediate features the layer can represent. The hidden representation is not manually specified; training discovers feature combinations that make the final task easier.

![A fully connected multilayer perceptron with four inputs, five hidden units, and three outputs.](assets/d2l-mlp.svg){fig-align="center" width="68%" fig-alt="A multilayer perceptron diagram with an input layer, one fully connected hidden layer, and an output layer."}

*Image source: [Dive into Deep Learning, Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

The activation is essential. If it is removed, two affine layers collapse into one:

$$
(XW_1^{\top}+\mathbf{b}_1)W_2^{\top}+\mathbf{b}_2
= X(W_2W_1)^{\top} + (\mathbf{b}_1W_2^{\top}+\mathbf{b}_2).
$$

Depth would then change the parameterization but not the family of input-output functions. A nonlinear $\phi$ prevents this algebraic collapse and lets the network construct context-dependent features. For ReLU networks, each activation pattern selects a local affine map; many patterns partition the input space into many piecewise-linear regions.

**Width** increases the number of features available at a layer. **Depth** composes features hierarchically, so a later unit can depend on patterns built by earlier units. Universal-approximation results show that a sufficiently wide one-hidden-layer MLP can approximate broad classes of functions on bounded domains, but this is an existence result, not a guarantee that optimization, data, or parameter efficiency will be favorable. Deep structure can often express compositional functions far more compactly.

<details>
<summary><strong>PyTorch: trace shape changes through a two-hidden-layer MLP</strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(11)

mlp = nn.Sequential(
    nn.Linear(4, 8),   # [B, 4] -> [B, 8]
    nn.ReLU(),
    nn.Linear(8, 6),   # [B, 8] -> [B, 6]
    nn.GELU(),
    nn.Linear(6, 3),   # [B, 6] -> [B, 3] logits
)

shape_trace = []
handles = []


def record_shape(name):
    """Create a forward hook that records a module's output contract."""
    def hook(module, inputs, output):
        shape_trace.append((name, tuple(output.shape)))
    return hook


for index, layer in enumerate(mlp):
    handles.append(layer.register_forward_hook(record_shape(f"{index}:{layer.__class__.__name__}")))

batch = torch.randn(5, 4)
logits = mlp(batch)

for handle in handles:
    handle.remove()

assert logits.shape == (5, 3)
print(*shape_trace, sep="\n")
~~~

</details>

MLPs are powerful but make few assumptions about input structure. Flattening a $224 \times 224$ RGB image and connecting it densely to 4,096 hidden units would require roughly $224 \cdot 224 \cdot 3 \cdot 4096 \approx 617$ million weights in the first layer alone. Convolutions, attention, and graph operators reduce or reorganize this cost by encoding locality, sharing, or relational structure. MLPs nevertheless remain central as prediction heads and channel-wise feed-forward blocks.

**Application.** MLPs are strong baselines for tabular data, low-dimensional signals, learned feature vectors, and classification heads. In Transformers, the feed-forward sublayer applies the same small MLP independently at every token position, mixing channels after attention has mixed information across positions.

**Comparison summary.** A linear model creates one affine map; an MLP composes affine maps with nonlinearities. Width supplies parallel features, depth composes them, and architecture-specific layers add useful inductive bias when dense all-to-all mixing would be inefficient.

### **Activation Functions** {#activation-functions}

An **activation function** transforms a pre-activation into the signal passed forward. Hidden-layer activations must usually be nonlinear; otherwise the entire stack remains affine. Their shape is normally unchanged because the function is applied elementwise:

$$
Z \in \mathbb{R}^{B \times D}
\xrightarrow{\phi}
H \in \mathbb{R}^{B \times D}.
$$

The choice affects expressiveness, gradient flow, sparsity, numerical range, and computational cost. It is helpful to distinguish hidden activations from output links: ReLU, GELU, and SiLU are common inside a network, while sigmoid or softmax may be implied by a task loss at the output.

![The ReLU function keeps positive inputs and maps negative inputs to zero.](assets/d2l-relu.svg){fig-align="center" width="64%" fig-alt="A graph of the rectified linear unit, which is zero for negative inputs and linear for positive inputs."}

*Image source: [Dive into Deep Learning, Activation Functions](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#activation-functions), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

| Activation | Definition | Range | Main behavior | Frequent use or caution |
|---|---|---|---|---|
| ReLU | $\max(0,x)$ | $[0,\infty)$ | cheap, sparse, slope 1 for positive inputs | a unit can become inactive if it stays negative |
| Leaky ReLU | $\max(x,\alpha x)$ | $(-\infty,\infty)$ | preserves a small negative slope | adds a slope hyperparameter |
| Sigmoid | $\sigma(x)=1/(1+e^{-x})$ | $(0,1)$ | interpretable gate or Bernoulli probability | saturates at large $|x|$ and is not zero-centered |
| Tanh | $\tanh(x)$ | $(-1,1)$ | zero-centered bounded signal | also saturates at large $|x|$ |
| GELU | $x\Phi(x)$ | approximately unbounded | smooth input-dependent attenuation | common in Transformer MLPs |
| SiLU / Swish | $x\sigma(x)$ | approximately unbounded | smooth and mildly non-monotonic | common in modern CNNs and gated MLPs |

ReLU has derivative 0 on the negative half-line and 1 on the positive half-line. Its exact derivative at zero is convention-dependent; frameworks choose a subgradient. Sigmoid satisfies $\sigma'(x)=\sigma(x)(1-\sigma(x))$, whose maximum is $1/4$, so repeated saturated sigmoid layers can attenuate gradients strongly. GELU and SiLU preserve small negative responses and change smoothly, but their extra arithmetic does not automatically make them best for every model or device.

Activation choice interacts with initialization and normalization. A variance-preserving initialization for ReLU-like layers uses a different scale from one designed for linear or tanh activations. Changing the activation while keeping every other assumption fixed can therefore alter both forward activation statistics and backward gradient statistics.

<details>
<summary><strong>PyTorch: compare activation values and local derivatives</strong></summary>

~~~python
import torch
from torch.nn import functional as F

sample_points = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
activation_functions = {
    "relu": F.relu,
    "leaky_relu": lambda x: F.leaky_relu(x, negative_slope=0.1),
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "gelu": F.gelu,
    "silu": F.silu,
}

for name, function in activation_functions.items():
    x = sample_points.clone().requires_grad_(True)
    y = function(x)

    # Summing asks autograd for dy_i / dx_i at every independent element.
    y.sum().backward()
    print(f"{name:11s} values={y.detach().round(decimals=3).tolist()}")
    print(f"{'':11s} slopes={x.grad.round(decimals=3).tolist()}")

# A negative ReLU input has no local gradient, while Leaky ReLU keeps a path.
negative = torch.tensor([-2.0], requires_grad=True)
F.relu(negative).backward()
relu_slope = negative.grad.item()

negative.grad.zero_()
F.leaky_relu(negative, negative_slope=0.1).backward()
leaky_slope = negative.grad.item()

assert relu_slope == 0.0
assert abs(leaky_slope - 0.1) < 1e-6
~~~

</details>

**Application.** ReLU remains a dependable default in many convolutional and simple MLP models. GELU is common in BERT-style Transformers, while SiLU and SwiGLU-like mechanisms appear in many modern architectures. Sigmoid is especially valuable as a gate or binary-output link, not as a universal hidden-layer default.

**Comparison summary.** ReLU is simple and sparse; Leaky ReLU protects a negative gradient path; sigmoid and tanh bound signals but saturate; GELU and SiLU provide smooth input-dependent attenuation. Select an activation together with initialization, normalization, architecture, and deployment constraints rather than by popularity alone.

### **Embeddings and Learned Representations** {#embeddings-learned-representations}

An **embedding layer** maps a discrete identifier to a dense learned vector. If a vocabulary contains $V$ items and each item receives a $D$-dimensional representation, the parameters form a table

$$
E \in \mathbb{R}^{V \times D}.
$$

For item index $i$, the output is row $E_i$. This operation is equivalent to multiplying a one-hot vector $\mathbf{e}_i \in \mathbb{R}^{V}$ by the table,

$$
\mathbf{h}_i = \mathbf{e}_i^{\top}E,
$$

but a lookup avoids constructing a mostly-zero vector and avoids multiplying by every row. For token IDs with shape `[B, L]`, `nn.Embedding(V, D)` returns `[B, L, D]`.

The important idea is not merely dimensionality reduction. An identifier has no meaningful arithmetic geometry: token 23 is not naturally closer to token 24 than to token 900. Training builds a continuous representation space in which task-relevant items can acquire similar directions or neighborhoods. The geometry is learned from the objective, so an embedding that is useful for sentiment may organize words differently from one trained for syntax or recommendation.

Embedding tables also appear outside language: user IDs, product IDs, categorical tabular features, graph nodes, image patches after linear projection, and time or position indices. A **padding index** can reserve a row that does not update. Rare and unseen categories require an explicit policy such as an unknown token, hashing, subword composition, or feature-based encoding.

<details>
<summary><strong>PyTorch: verify lookup equivalence and padding behavior</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(13)

VOCAB_SIZE, EMBED_DIM = 7, 4
embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
token_ids = torch.tensor([[1, 4, 0], [4, 2, 6]])  # [B=2, L=3]

lookup_vectors = embedding(token_ids)              # [2, 3, 4]
one_hot = F.one_hot(token_ids, num_classes=VOCAB_SIZE).float()
matrix_vectors = one_hot @ embedding.weight

assert lookup_vectors.shape == (2, 3, EMBED_DIM)
assert torch.allclose(lookup_vectors, matrix_vectors)
assert torch.equal(lookup_vectors[0, 1], lookup_vectors[1, 0])  # same ID, same row

# The padding row is excluded from gradient updates.
lookup_vectors.sum().backward()
assert torch.count_nonzero(embedding.weight.grad[0]) == 0
assert torch.count_nonzero(embedding.weight.grad[4]) > 0

print("token IDs:", tuple(token_ids.shape))
print("embedded sequence:", tuple(lookup_vectors.shape))
~~~

</details>

In language models, the input embedding matrix is sometimes **weight-tied** with the output vocabulary projection. Instead of learning two unrelated matrices of shape roughly `[V, D]`, one parameter table serves both input lookup and output scoring. This reduces parameters and connects the geometry used to read tokens with the geometry used to predict them, but only when dimensions and modeling assumptions are compatible.

**Application.** Embeddings allow a model to learn representations for words, subwords, users, products, classes, or positions jointly with the downstream task. They are especially effective when identities repeat often enough for their rows to receive informative updates.

**Comparison summary.** One-hot encoding preserves identity but is high-dimensional and has no learned similarity; an embedding lookup is compact and trainable. A lookup table memorizes item-specific vectors, while an encoder built from observable features can generalize to unseen items. Many systems combine both.

### **Output Heads** {#output-heads}

A network backbone produces a representation; an **output head** converts that representation into the parameterization required by a task. Separating the two clarifies transfer learning: one backbone can support multiple heads, and a new task may replace only the final head.

Suppose the backbone returns $H \in \mathbb{R}^{B \times D}$. Common contracts are:

| Task | Head output | Target | Typical training loss | Inference transform |
|---|---|---|---|---|
| Multiclass classification | logits `[B, K]` | class index `[B]` | cross-entropy | softmax, then argmax |
| Binary classification | logit `[B]` or `[B,1]` | binary float | BCE with logits | sigmoid and threshold |
| Multilabel classification | logits `[B, K]` | binary matrix `[B,K]` | elementwise BCE with logits | independent sigmoid thresholds |
| Regression | value `[B,R]` | continuous `[B,R]` | MSE, MAE, or likelihood loss | often identity |
| Token classification | logits `[B,L,K]` | token labels `[B,L]` | masked token cross-entropy | per-token argmax or structured decoder |
| Retrieval | embedding `[B,D_r]` | pairs or relevance labels | contrastive or ranking loss | similarity search |

**Logits** are unconstrained scores. Applying softmax before `CrossEntropyLoss`, or sigmoid before `BCEWithLogitsLoss`, is a common mistake because these losses already combine the link function with a numerically stable log-loss calculation. Probability conversion belongs at inference or interpretation time unless a different loss explicitly requires probabilities.

An output head encodes assumptions. A $K$-class softmax assumes one mutually exclusive outcome; $K$ sigmoid outputs allow several labels simultaneously. A scalar regression head assumes the response is adequately represented by a point estimate, while a probabilistic head may predict a mean and scale, quantiles, or distribution parameters to express uncertainty.

<details>
<summary><strong>PyTorch: attach classification and regression heads to one representation</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class MultiTaskHeads(nn.Module):
    """Map one [B, D] representation to two task-specific outputs."""

    def __init__(self, feature_dim: int, number_of_classes: int):
        super().__init__()
        self.classifier = nn.Linear(feature_dim, number_of_classes)
        self.regressor = nn.Linear(feature_dim, 1)

    def forward(self, representation: torch.Tensor):
        return {
            "class_logits": self.classifier(representation),  # [B, K]
            "value": self.regressor(representation).squeeze(-1),  # [B]
        }


torch.manual_seed(17)
B, D, K = 6, 12, 4
representation = torch.randn(B, D)
heads = MultiTaskHeads(D, K)
predictions = heads(representation)

class_targets = torch.tensor([0, 3, 1, 2, 1, 0])
value_targets = torch.randn(B)

# Pass raw logits to cross-entropy; do not apply softmax first.
classification_loss = F.cross_entropy(predictions["class_logits"], class_targets)
regression_loss = F.mse_loss(predictions["value"], value_targets)
total_loss = classification_loss + 0.25 * regression_loss

assert predictions["class_logits"].shape == (B, K)
assert predictions["value"].shape == (B,)
assert total_loss.ndim == 0
~~~

</details>

The relative weighting of multiple head losses affects which task dominates shared features. Equal numeric weights do not imply equal gradient influence because the losses can have different scales, noise, and curvature. Multi-task systems therefore monitor each task separately and may tune, normalize, or dynamically adapt loss weights.

**Application.** A vision backbone can support category, bounding-box, and segmentation heads; a language encoder can support intent and token-label heads; a recommendation model can predict click probability and expected value. The head should expose exactly the quantities consumed by the loss and evaluation protocol.

**Comparison summary.** The backbone learns reusable features; the head imposes a task contract. Softmax expresses competition among classes, sigmoid treats labels independently, regression predicts continuous quantities, and embedding heads optimize geometry rather than direct labels.

### **Residual and Skip Connections** {#residual-skip-connections}

A **skip connection** creates a path that bypasses one or more transformations. The most common residual block computes

$$
Y = X + F(X;\theta),
$$

where $F$ is a learned residual branch and the shortcut is the identity. Instead of requiring the branch to reproduce the entire desired mapping $H(X)$, the block parameterizes the change $F(X)=H(X)-X$. If preserving the current representation is useful, the branch can approach zero while the identity path remains available.

![The original ResNet residual block adds an identity shortcut to a learned two-layer residual branch.](assets/resnet-residual-block.png){fig-align="center" width="58%" fig-alt="The original ResNet residual building block, with two weight layers on one branch and an identity shortcut added before ReLU."}

*Image source: He et al., [Deep Residual Learning for Image Recognition](https://openaccess.thecvf.com/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html), Figure 2.*

Addition requires compatible shapes. If $F(X)$ changes width, channel count, or spatial resolution, the shortcut must be transformed:

$$
Y = P(X) + F(X),
$$

where $P$ may be a learned linear projection, a strided convolution, or a deterministic padding/downsampling operation. An identity shortcut adds no parameters; a projection shortcut does.

Residual paths help optimization because they create a direct route for both activations and sensitivity signals. Informally, differentiating $Y=X+F(X)$ with respect to $X$ contains an identity term in addition to the derivative through $F$. Chapter 04 will derive this precisely. Residual connections do not guarantee stable training by themselves: poor scaling, mismatched normalization, or very large residual updates can still destabilize a deep stack.

The placement of normalization and activation matters. In a **post-activation** block, transformation, addition, and activation follow the original ResNet pattern. In many modern **pre-normalization** blocks, the residual branch first normalizes its input and the addition is left as a clean identity update. This changes signal propagation even when tensor shapes are identical.

<details>
<summary><strong>PyTorch: implement an MLP residual block with an optional projection</strong></summary>

~~~python
import torch
from torch import nn


class ResidualMLPBlock(nn.Module):
    """A pre-normalized residual block for [B, D] feature tensors."""

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.branch = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.shortcut = (
            nn.Identity()
            if input_dim == output_dim
            else nn.Linear(input_dim, output_dim, bias=False)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual_update = self.branch(self.norm(x))
        return self.shortcut(x) + residual_update


same_width = ResidualMLPBlock(input_dim=16, output_dim=16, hidden_dim=32)
changed_width = ResidualMLPBlock(input_dim=16, output_dim=24, hidden_dim=32)
features = torch.randn(8, 16)

assert same_width(features).shape == (8, 16)
assert changed_width(features).shape == (8, 24)
assert sum(p.numel() for p in same_width.shortcut.parameters()) == 0
assert sum(p.numel() for p in changed_width.shortcut.parameters()) == 16 * 24
~~~

</details>

Not every skip connection uses addition. U-Net transfers encoder feature maps to a decoder by concatenation, DenseNet concatenates many earlier representations, and encoder-decoder models can route multiscale information across a bottleneck. Concatenation preserves both inputs but increases width and downstream cost; addition preserves width but forces the branches into a shared coordinate system.

**Application.** Residual updates are standard in ResNets, Transformers, diffusion networks, state-space blocks, and large MLP architectures. Cross-scale skip connections are especially important when a bottleneck would otherwise discard fine spatial detail.

**Comparison summary.** Plain stacking replaces the representation at every block; residual addition learns a same-shaped update; projection residuals reconcile dimensions; concatenative skips preserve separate feature sets at the cost of larger tensors.

### **Gating Mechanisms** {#gating-mechanisms}

A **gate** is a learned, data-dependent control over information flow. A generic interpolation gate is

$$
G = \sigma(A(X)),
\qquad
Y = G \odot U(X) + (1-G) \odot V(X),
$$

where $G$ has values in $(0,1)$ and $\odot$ denotes elementwise multiplication. A gate is analogous to a continuously adjustable valve: it can retain one stream, favor another, or mix them differently for every example, token, channel, or spatial position.

Gating is more expressive than a fixed residual sum because the routing strength depends on the input. It also introduces additional projections and a new failure mode: sigmoid gates can saturate near 0 or 1, producing small derivatives and making a routing decision difficult to change. Gate bias initialization is sometimes chosen so a network initially preserves information rather than suppressing it.

Several widely used blocks fit this pattern:

| Mechanism | Formula | Interpretation |
|---|---|---|
| GLU | $\operatorname{GLU}(X)=A(X)\odot\sigma(B(X))$ | sigmoid branch controls a value branch |
| SwiGLU | $\operatorname{SwiGLU}(X)=A(X)\odot\operatorname{SiLU}(B(X))$ | smooth gated feed-forward transformation |
| Highway update | $T(X)\odot H(X)+(1-T(X))\odot X$ | choose transformed versus carried information |
| LSTM/GRU gates | multiple sigmoid-controlled state updates | decide what sequence state to write, erase, or expose |
| Mixture-of-experts router | normalized scores over expert branches | choose sparse or dense computation paths |

The word *gate* does not imply that values are exactly binary. During ordinary differentiable training, the controls are usually continuous. Hard or sparse routing needs additional estimators, regularizers, or dispatch logic.

<details>
<summary><strong>PyTorch: implement a SwiGLU feed-forward block</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class SwiGLU(nn.Module):
    """A gated feed-forward block that preserves the final feature width."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        # One projection produces both the value and gate pre-activations.
        self.value_and_gate = nn.Linear(input_dim, 2 * hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        value, gate = self.value_and_gate(x).chunk(2, dim=-1)
        hidden = value * F.silu(gate)
        return self.output(hidden)


torch.manual_seed(19)
B, L, D = 2, 5, 16
block = SwiGLU(input_dim=D, hidden_dim=32, output_dim=D)
sequence = torch.randn(B, L, D)
output = block(sequence)

assert output.shape == (B, L, D)
assert block.value_and_gate.weight.shape == (64, D)
print("parameters:", sum(parameter.numel() for parameter in block.parameters()))
~~~

</details>

**Application.** Gates manage memory in recurrent networks, regulate feature flow in highway networks, improve Transformer feed-forward layers, and route tokens through mixture-of-experts systems. Their usefulness is greatest when the model should decide *how much* or *which route* rather than always applying the same update.

**Comparison summary.** An activation transforms a signal; a residual connection adds a fixed structural path; a gate learns a data-dependent multiplier or mixture. Gating provides adaptive control but costs parameters and can saturate or collapse without suitable initialization and monitoring.

### **Normalization Layers** {#normalization-layers}

A **normalization layer** standardizes selected groups of activations and usually follows with learned scale and shift parameters. For a set of values $S$ chosen by the normalization rule,

$$
\mu_S=\frac{1}{|S|}\sum_{i\in S}x_i,
\qquad
\sigma_S^2=\frac{1}{|S|}\sum_{i\in S}(x_i-\mu_S)^2,
$$

$$
\widehat{x}_i=\frac{x_i-\mu_S}{\sqrt{\sigma_S^2+\epsilon}},
\qquad
y_i=\gamma_i\widehat{x}_i+\beta_i.
$$

The core difference among normalization methods is the definition of $S$: which examples, channels, positions, or spatial locations share a mean and variance. The small $\epsilon$ prevents division by zero, while learnable $\gamma$ and $\beta$ let the network restore or reshape useful scales after standardization.

![Batch, layer, instance, and group normalization aggregate different subsets of a feature-map tensor.](assets/normalization-axes-comparison.png){fig-align="center" width="88%" fig-alt="A comparison of Batch Normalization, Layer Normalization, Instance Normalization, and Group Normalization, with blue cells showing values normalized together."}

*Image source: Wu and He, [Group Normalization](https://openaccess.thecvf.com/content_ECCV_2018/html/Yuxin_Wu_Group_Normalization_ECCV_2018_paper.html), Figure 2.*

For an image tensor `[B, C, H, W]`:

- **BatchNorm** computes one set of statistics per channel over `[B, H, W]`. During evaluation it normally uses running estimates collected during training, so train/eval mode matters.
- **LayerNorm** computes statistics within each example over selected feature axes. In sequence models with `[B, L, D]`, it usually normalizes the final width `D` independently for each token.
- **InstanceNorm** computes statistics per example and channel over `[H, W]`; it is common where instance-specific contrast should be normalized.
- **GroupNorm** divides channels into groups and normalizes each group within each example. It does not depend on batch statistics.
- **RMSNorm** rescales by root mean square without subtracting the mean, reducing arithmetic while preserving a related stabilizing effect.

Normalization is not the same as input preprocessing. Dataset normalization uses fixed statistics to make raw features comparable. An internal normalization layer acts on changing learned activations, participates in the model graph, and may contain trainable parameters or running buffers.

<details>
<summary><strong>PyTorch: verify the axes normalized by BN, LN, IN, and GN</strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(23)
B, C, H, W = 4, 6, 3, 3
x = torch.randn(B, C, H, W) * 3.0 + 5.0

# Disable affine parameters so the post-normalization means are easy to inspect.
batch_norm = nn.BatchNorm2d(C, affine=False, track_running_stats=False)
layer_norm = nn.LayerNorm((C, H, W), elementwise_affine=False)
instance_norm = nn.InstanceNorm2d(C, affine=False, track_running_stats=False)
group_norm = nn.GroupNorm(num_groups=3, num_channels=C, affine=False)

bn_output = batch_norm(x)
ln_output = layer_norm(x)
in_output = instance_norm(x)
gn_output = group_norm(x)

# Each assertion averages over exactly the axes used to estimate the mean.
assert torch.allclose(bn_output.mean(dim=(0, 2, 3)), torch.zeros(C), atol=1e-5)
assert torch.allclose(ln_output.mean(dim=(1, 2, 3)), torch.zeros(B), atol=1e-5)
assert torch.allclose(in_output.mean(dim=(2, 3)), torch.zeros(B, C), atol=1e-5)

grouped = gn_output.reshape(B, 3, C // 3, H, W)
assert torch.allclose(grouped.mean(dim=(2, 3, 4)), torch.zeros(B, 3), atol=1e-5)

print("BN means per channel:", bn_output.mean(dim=(0, 2, 3)).round(decimals=6))
print("GN means per sample/group:", grouped.mean(dim=(2, 3, 4)).round(decimals=6))
~~~

</details>

BatchNorm can work extremely well with representative, sufficiently large batches, but small or non-independent batches produce noisy statistics and a mismatch can arise between training and inference. LayerNorm and GroupNorm avoid cross-example dependence, making them natural for variable-length sequence models, small-batch vision, and distributed settings where synchronizing batch statistics would be expensive.

**Application.** BatchNorm is deeply associated with convolutional networks; LayerNorm and RMSNorm dominate Transformer-like architectures; GroupNorm is useful in detection, segmentation, diffusion, and other small-batch vision workloads; InstanceNorm is common in style-related image generation.

**Comparison summary.** All normalization layers rescale activations, but they couple different axes and maintain different state. BatchNorm depends on the batch and has distinct train/eval behavior; LayerNorm, GroupNorm, and InstanceNorm use per-example statistics; RMSNorm omits mean centering.

### **Parameter Counting and Computational Cost** {#parameter-counting-computational-cost}

Parameter count measures learned storage, while computational cost measures work performed for particular input shapes. They are related but not interchangeable. An embedding table may contain many parameters while touching only a few rows per example; a parameter-free activation can still process every element of a large activation tensor.

For a dense layer from $D_{in}$ to $D_{out}$,

$$
N_{params}=D_{out}D_{in}+D_{out}
$$

when bias is enabled. Processing $N$ input vectors requires approximately

$$
N \cdot D_{in}D_{out}
$$

multiply-accumulate operations (MACs), plus lower-order bias and activation work. Some reports count one MAC as one operation; others count multiplication and addition separately as two FLOPs. A resource claim must state its convention.

Useful first-order formulas include:

| Block | Trainable parameters | Dominant per-vector computation |
|---|---:|---:|
| Linear $D_{in}\to D_{out}$ | $D_{in}D_{out}+D_{out}$ | $D_{in}D_{out}$ MACs |
| Embedding $V\times D$ | $VD$ | lookup of $D$ values per ID |
| LayerNorm over $D$ | normally $2D$ | reductions and elementwise work in $O(D)$ |
| Two-layer MLP $D\to H\to D$ | about $2DH$ | about $2DH$ MACs |
| SwiGLU $D\to 2H$, then $H\to D$ | about $3DH$ | about $3DH$ MACs |

Batch size and sequence length multiply computation and activation memory but do not change model parameter count. Training memory additionally includes gradients, optimizer state, saved activations, and temporary workspaces. A small parameter reduction may have little effect if long-sequence activations dominate memory; conversely, a huge embedding table can dominate checkpoint size without dominating FLOPs.

<details>
<summary><strong>PyTorch: count parameters and estimate Linear-layer MACs with hooks</strong></summary>

~~~python
import torch
from torch import nn


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


def estimate_linear_macs(module: nn.Module, sample: torch.Tensor) -> int:
    """Estimate only dense Linear MACs for one forward pass."""
    total_macs = 0
    handles = []

    def linear_hook(layer, inputs, output):
        nonlocal total_macs
        input_tensor = inputs[0]
        number_of_vectors = input_tensor.numel() // layer.in_features
        total_macs += number_of_vectors * layer.in_features * layer.out_features

    for child in module.modules():
        if isinstance(child, nn.Linear):
            handles.append(child.register_forward_hook(linear_hook))

    with torch.no_grad():
        module(sample)

    for handle in handles:
        handle.remove()
    return total_macs


network = nn.Sequential(
    nn.Linear(64, 128),
    nn.GELU(),
    nn.Linear(128, 10),
)
batch = torch.randn(32, 64)

parameter_count = count_trainable_parameters(network)
linear_macs = estimate_linear_macs(network, batch)
parameter_mebibytes_fp32 = parameter_count * 4 / 2**20

assert parameter_count == (64 * 128 + 128) + (128 * 10 + 10)
assert linear_macs == 32 * (64 * 128 + 128 * 10)

print("trainable parameters:", parameter_count)
print("Linear MACs for the batch:", linear_macs)
print("parameter payload in fp32 MiB:", round(parameter_mebibytes_fp32, 4))
~~~

</details>

Hook-based estimates are educational approximations, not complete profilers. They omit activations, normalization, memory traffic, kernel fusion, parallel efficiency, and hardware-specific behavior. Real latency should be measured after warm-up on the target device with representative shapes; peak memory should be observed under the actual training or inference mode.

**Application.** Parameter formulas catch accidental oversized heads or embeddings before training. MAC and activation estimates guide width, depth, batch size, and sequence-length choices. Profiling then determines whether the theoretical bottleneck is visible on the target hardware.

**Comparison summary.** Parameters describe learned state; MACs/FLOPs describe arithmetic; activation size and memory traffic influence runtime and memory; wall-clock latency reflects the complete software-hardware system. Report all with shapes and counting conventions.

### **Building a Network from Reusable PyTorch Modules** {#building-network-reusable-pytorch-modules}

A reusable neural network module should own one coherent transformation and make its tensor contract obvious. Good module boundaries provide:

- explicit input and output widths;
- registered parameters, buffers, and child modules;
- a forward method that contains data flow rather than training-loop policy;
- shape-preserving blocks that can be stacked safely;
- interchangeable heads for different tasks;
- names that remain meaningful in checkpoints, profiles, and error traces.

`nn.Sequential` is ideal for one straight pipeline. `nn.ModuleList` registers a variable-length collection when the forward pass needs an explicit loop. `nn.ModuleDict` registers named alternatives or heads. A plain Python list or dictionary does not automatically register contained modules, so its parameters may be absent from `.parameters()`, device movement, and state dictionaries.

The following small architecture combines the chapter's components. A stem projects raw features to a shared width. Each residual block uses pre-normalization and a SwiGLU residual update. Named heads turn the final representation into classification and regression outputs.

<details>
<summary><strong>PyTorch: compose a reusable residual backbone with multiple heads</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class GatedResidualBlock(nn.Module):
    """Pre-norm SwiGLU residual block with contract [B, D] -> [B, D]."""

    def __init__(self, width: int, hidden_width: int, dropout: float = 0.0):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.expand = nn.Linear(width, 2 * hidden_width)
        self.contract = nn.Linear(hidden_width, width)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm(x)
        value, gate = self.expand(normalized).chunk(2, dim=-1)
        update = self.contract(value * F.silu(gate))
        return x + self.dropout(update)


class MultiTaskNetwork(nn.Module):
    """Shared feature backbone with named task-specific output heads."""

    def __init__(
        self,
        input_dim: int,
        width: int,
        hidden_width: int,
        depth: int,
        number_of_classes: int,
    ):
        super().__init__()
        self.stem = nn.Linear(input_dim, width)
        self.blocks = nn.ModuleList(
            [GatedResidualBlock(width, hidden_width, dropout=0.1) for _ in range(depth)]
        )
        self.final_norm = nn.LayerNorm(width)
        self.heads = nn.ModuleDict(
            {
                "class_logits": nn.Linear(width, number_of_classes),
                "value": nn.Linear(width, 1),
            }
        )

    def forward(self, features: torch.Tensor):
        hidden = self.stem(features)                  # [B, D_in] -> [B, D]
        for block in self.blocks:
            hidden = block(hidden)                    # width is preserved
        representation = self.final_norm(hidden)
        return {
            "representation": representation,
            "class_logits": self.heads["class_logits"](representation),
            "value": self.heads["value"](representation).squeeze(-1),
        }


torch.manual_seed(29)
model = MultiTaskNetwork(
    input_dim=20,
    width=32,
    hidden_width=64,
    depth=3,
    number_of_classes=5,
)
batch = torch.randn(7, 20)
outputs = model(batch)

assert outputs["representation"].shape == (7, 32)
assert outputs["class_logits"].shape == (7, 5)
assert outputs["value"].shape == (7,)

# Registered submodules appear automatically in parameters and checkpoints.
state = model.state_dict()
assert "blocks.0.expand.weight" in state
assert "heads.class_logits.weight" in state
print("trainable parameters:", sum(p.numel() for p in model.parameters()))
~~~

</details>

The architecture is intentionally task-agnostic. It does not include an optimizer, loss, batch loader, or threshold because those belong to the training and evaluation system. The forward method returns raw logits and an explicit representation, making downstream behavior testable without hidden post-processing.

Useful module tests include shape checks across several batch sizes, deterministic behavior in evaluation mode, finite outputs for expected ranges, complete state-dictionary round trips, and confirmation that every intended parameter receives a gradient after a toy loss. Chapter 04 will examine that final check through automatic differentiation.

**Application.** Reusable modules support ablation studies, transfer learning, multiple output heads, architecture search, profiling, checkpoint migration, and team collaboration. A block can be replaced only when its input-output contract remains compatible.

**Comparison summary.** `Sequential` expresses a straight chain, `ModuleList` registers modules used by custom control flow, and `ModuleDict` registers named branches. Good composition separates architecture from training policy and makes shapes, state, and task heads inspectable.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Neural networks become easier to reason about when every architecture is decomposed into three roles: **representation transformation**, **information-flow control**, and **task interpretation**.

| Building block | Primary role | Preserves shape? | Learned state | Main failure to inspect |
|---|---|---:|---:|---|
| Linear layer | mix and resize features | only when widths match | weight and optional bias | wrong axis or excessive parameter growth |
| MLP | compose nonlinear feature transformations | configurable | multiple dense layers | missing activation or unsuitable width/depth |
| Activation | add nonlinearity or bounded control | normally yes | usually none | saturation, inactive units, mismatched initialization |
| Embedding | map identity to learned geometry | adds feature axis | lookup table | unseen items, padding, oversized vocabulary |
| Output head | enforce task-specific prediction contract | task-dependent | projection parameters | incorrect logits, targets, or loss pairing |
| Residual connection | preserve and update information | yes unless projected | none or projection | incompatible shapes or poorly scaled update |
| Gate | route or modulate information | usually yes | gate/value projections | saturation or routing collapse |
| Normalization | standardize selected activation groups | yes | scale/shift; sometimes running state | normalizing wrong axes or train/eval mismatch |

The main conclusions are:

1. A dense neuron computes an affine feature combination; the nonlinear activation is what prevents a deep stack from collapsing into one affine map.
2. MLP width controls parallel feature capacity, while depth composes features into increasingly structured representations.
3. Activation functions alter both forward signal statistics and local derivative behavior, so they must match initialization and architecture.
4. Embeddings turn discrete IDs into trainable geometry and are efficient because lookup replaces explicit one-hot multiplication.
5. Output heads should emit the exact raw quantities expected by the loss; logits and probabilities are not interchangeable.
6. Residual paths provide an explicit information-preserving route, while gates make routing strength data-dependent.
7. Normalization methods differ primarily in which axes share statistics and whether behavior depends on the training batch.
8. Parameter count, arithmetic, activation memory, and measured latency answer different resource questions.
9. Reusable PyTorch modules should make shape contracts and registered state visible while keeping training policy outside `forward`.

The next chapter explains how these blocks learn: local derivatives are composed through the computation graph, reverse-mode automatic differentiation propagates credit, and gradient checks reveal broken or numerically unstable paths.